# Chunking Refinado — PDFs → Markdown → Busca Semântica

Este notebook:
1. Recebe **PDFs** por upload e os converte para **Markdown** com `pymupdf4llm`
2. Compara **10 estratégias de chunking** com overlaps calibrados
3. Executa busca semântica com **3 queries** temáticas
4. Exibe **tabela comparativa** e **gráfico de barras**

| # | Estratégia | Variável |
|---|-----------|----------|
| 1 | Fixo 256, sem overlap | baseline mínimo |
| 2 | Fixo 512, sem overlap | tamanho padrão RAG |
| 3 | Fixo 1024, sem overlap | bloco maior |
| 4 | Fixo 512, overlap 64 (12%) | overlap leve |
| 5 | Fixo 512, overlap 128 (25%) | overlap moderado |
| 6 | Fixo 512, overlap 256 (50%) | overlap pesado |
| 7 | Recursivo 512, overlap 64 | recursivo leve |
| 8 | Recursivo 512, overlap 128 | recursivo moderado |
| 9 | Recursivo 1024, overlap 128 | recursivo grande |
| 10 | Por seção/heading Markdown | estrutura semântica |

---
### 1. Instalação de Bibliotecas

In [ ]:
!pip install -q openai langchain-text-splitters pymupdf4llm pandas matplotlib

---
### 2. Configuração da API

In [ ]:
import time
import re
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymupdf4llm
from google.colab import userdata, files
from openai import OpenAI
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get('OPENROUTER_API_KEY')
)

MAX_CHARS = 6000
EMBEDDING_MODEL = 'openai/text-embedding-3-small'

QUERIES = [
    'O que é o mecanismo de atenção (attention mechanism) em modelos de linguagem?',
    'Como funciona o fine-tuning com RLHF (Reinforcement Learning from Human Feedback)?',
    'O que são scaling laws e como elas influenciam o desempenho de LLMs?',
]

print('Configuração concluída!')

---
### 3. Upload dos PDFs e Conversão para Markdown

In [ ]:
def limpar_texto(texto):
    """Remove hifens de quebra de linha e linhas muito curtas."""
    texto = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', texto)
    linhas = [l for l in texto.splitlines() if len(l.strip()) > 4 or l.strip() == '']
    return '\n'.join(linhas)

print('Faça upload dos arquivos PDF:')
uploaded = files.upload()

documentos_md = []
for nome_arquivo, conteudo in uploaded.items():
    print(f'Convertendo {nome_arquivo}...', end=' ')
    try:
        # Salva o PDF temporariamente
        caminho_tmp = f'/tmp/{nome_arquivo}'
        with open(caminho_tmp, 'wb') as f:
            f.write(conteudo)
        # Converte para Markdown
        md_text = pymupdf4llm.to_markdown(caminho_tmp)
        md_text = limpar_texto(md_text)
        titulo = nome_arquivo.replace('.pdf', '').replace('_', ' ').title()
        documentos_md.append(f'# {titulo}\n\n{md_text}')
        print(f'OK ({len(md_text):,} chars)')
    except Exception as e:
        print(f'ERRO: {e}')

texto_completo = '\n\n'.join(documentos_md)
print(f'\nTotal: {len(documentos_md)} documentos, {len(texto_completo):,} caracteres')

---
### 4. Funções Auxiliares

In [ ]:
def get_embedding(texto):
    texto = str(texto)[:MAX_CHARS]
    resp = client.embeddings.create(input=texto, model=EMBEDDING_MODEL)
    return resp.data[0].embedding

def similaridade_cosseno(a, b):
    a, b = np.array(a), np.array(b)
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return 0.0 if na == 0 or nb == 0 else float(np.dot(a, b) / (na * nb))

def busca_semantica(query, chunks, top_k=3):
    vec_q = get_embedding(query)
    resultados = []
    batch_size = 10
    for i in range(0, len(chunks), batch_size):
        lote = [str(c)[:MAX_CHARS] for c in chunks[i:i+batch_size]]
        resp = client.embeddings.create(input=lote, model=EMBEDDING_MODEL)
        for j, dado in enumerate(resp.data):
            sim = similaridade_cosseno(vec_q, dado.embedding)
            resultados.append({'Trecho': chunks[i+j], 'Similaridade': sim})
        if i + batch_size < len(chunks):
            time.sleep(1)
    resultados.sort(key=lambda x: x['Similaridade'], reverse=True)
    return resultados[:top_k]

def estatisticas_chunks(chunks):
    tamanhos = [len(str(c)) for c in chunks]
    if not tamanhos:
        return {'total': 0, 'media': 0, 'min': 0, 'max': 0}
    return {'total': len(chunks), 'media': int(np.mean(tamanhos)), 'min': min(tamanhos), 'max': max(tamanhos)}

print('Funções auxiliares prontas!')

---
### 5. Definição das 10 Estratégias Calibradas

In [ ]:
estrategias = [
    {'grupo': 1, 'tipo': 'texto',
     'nome': 'Fixo 256, sem overlap', 'variavel': 'baseline mínimo',
     'splitter': CharacterTextSplitter(separator='', chunk_size=256, chunk_overlap=0)},
    {'grupo': 2, 'tipo': 'texto',
     'nome': 'Fixo 512, sem overlap', 'variavel': 'tamanho padrão RAG',
     'splitter': CharacterTextSplitter(separator='', chunk_size=512, chunk_overlap=0)},
    {'grupo': 3, 'tipo': 'texto',
     'nome': 'Fixo 1024, sem overlap', 'variavel': 'bloco maior',
     'splitter': CharacterTextSplitter(separator='', chunk_size=1024, chunk_overlap=0)},
    {'grupo': 4, 'tipo': 'texto',
     'nome': 'Fixo 512, overlap 64 (12%)', 'variavel': 'overlap leve',
     'splitter': CharacterTextSplitter(separator='', chunk_size=512, chunk_overlap=64)},
    {'grupo': 5, 'tipo': 'texto',
     'nome': 'Fixo 512, overlap 128 (25%)', 'variavel': 'overlap moderado',
     'splitter': CharacterTextSplitter(separator='', chunk_size=512, chunk_overlap=128)},
    {'grupo': 6, 'tipo': 'texto',
     'nome': 'Fixo 512, overlap 256 (50%)', 'variavel': 'overlap pesado',
     'splitter': CharacterTextSplitter(separator='', chunk_size=512, chunk_overlap=256)},
    {'grupo': 7, 'tipo': 'texto',
     'nome': 'Recursivo 512, overlap 64', 'variavel': 'recursivo leve',
     'splitter': RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64)},
    {'grupo': 8, 'tipo': 'texto',
     'nome': 'Recursivo 512, overlap 128', 'variavel': 'recursivo moderado',
     'splitter': RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=128)},
    {'grupo': 9, 'tipo': 'texto',
     'nome': 'Recursivo 1024, overlap 128', 'variavel': 'recursivo grande',
     'splitter': RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=128)},
    {'grupo': 10, 'tipo': 'markdown',
     'nome': 'Por seção/heading Markdown', 'variavel': 'estrutura semântica',
     'splitter': MarkdownHeaderTextSplitter(headers_to_split_on=[('#', 'H1'), ('##', 'H2'), ('###', 'H3')])},
]
print(f'{len(estrategias)} estratégias definidas!')

---
### 6. Execução das 10 Estratégias

In [ ]:
resultados_finais = []

for est in estrategias:
    grupo = est['grupo']
    nome = est['nome']
    variavel = est['variavel']
    splitter = est['splitter']

    print('=' * 70)
    print(f'GRUPO {grupo}: {nome}')
    print(f'Variável: {variavel}')

    try:
        if est['tipo'] == 'markdown':
            docs = splitter.split_text(texto_completo)
            chunks = [d.page_content for d in docs if d.page_content.strip()]
        else:
            chunks = [c for c in splitter.split_text(texto_completo) if c.strip()]
    except Exception as e:
        print(f'ERRO ao dividir: {e}')
        continue

    chunks = [c for c in chunks if len(c.strip()) >= 20]
    stats = estatisticas_chunks(chunks)
    print(f'Chunks: {stats["total"]} | Média: {stats["media"]} chars | Min: {stats["min"]} | Max: {stats["max"]}')

    sims = []
    for qi, query in enumerate(QUERIES, 1):
        print(f'  Query {qi}: {query[:60]}...')
        try:
            top = busca_semantica(query, chunks, top_k=3)
            melhor = top[0]['Similaridade'] if top else 0.0
            sims.append(melhor)
            for idx, r in enumerate(top, 1):
                preview = ' '.join(r['Trecho'].split())[:150]
                print(f'    TOP {idx} (sim={r["Similaridade"]:.4f}): {preview}...')
        except Exception as e:
            print(f'    ERRO: {e}')
            sims.append(0.0)

    resultados_finais.append({
        'Grupo': grupo,
        'Estratégia': nome,
        'Variável': variavel,
        'Chunks': stats['total'],
        'Tam.Médio': stats['media'],
        'Sim Q1': round(sims[0], 4) if len(sims) > 0 else 0.0,
        'Sim Q2': round(sims[1], 4) if len(sims) > 1 else 0.0,
        'Sim Q3': round(sims[2], 4) if len(sims) > 2 else 0.0,
        'Média': round(np.mean(sims), 4) if sims else 0.0,
    })
    print()

print('Todas as estratégias executadas!')

---
### 7. Tabela Comparativa Final

In [ ]:
df = pd.DataFrame(resultados_finais).sort_values('Média', ascending=False)
df

---
### 8. Melhor Estratégia

In [ ]:
melhor = df.iloc[0]
print('=== Melhor Estratégia ===')
print(f'Grupo {int(melhor["Grupo"])}: {melhor["Estratégia"]}')
print(f'Variável: {melhor["Variável"]}')
print(f'Chunks: {int(melhor["Chunks"])} | Tam. Médio: {int(melhor["Tam.Médio"])} chars')
print(f'Sim Q1: {melhor["Sim Q1"]:.4f} | Sim Q2: {melhor["Sim Q2"]:.4f} | Sim Q3: {melhor["Sim Q3"]:.4f}')
print(f'Média Geral: {melhor["Média"]:.4f}')

---
### 9. Gráfico Comparativo

In [ ]:
df_plot = pd.DataFrame(resultados_finais).sort_values('Grupo')
x = np.arange(len(df_plot))
w = 0.2
cores = ['#4285F4', '#EA4335', '#FBBC04', '#34A853']

fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(x - 1.5*w, df_plot['Sim Q1'], w, label='Q1 – Atenção',      color=cores[0])
ax.bar(x - 0.5*w, df_plot['Sim Q2'], w, label='Q2 – RLHF',         color=cores[1])
ax.bar(x + 0.5*w, df_plot['Sim Q3'], w, label='Q3 – Scaling Laws',  color=cores[2])
ax.bar(x + 1.5*w, df_plot['Média'],  w, label='Média',              color=cores[3])

ax.set_xticks(x)
ax.set_xticklabels([f'G{int(g)}' for g in df_plot['Grupo']])
ax.set_xlabel('Estratégia de Chunking')
ax.set_ylabel('Similaridade Cosseno')
ax.set_title('Chunking Refinado — 10 Estratégias com Overlap Calibrado')
ax.set_ylim(0, 1)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()